In [ ]:
!pip install opencv-python roboflow imageio imageio-ffmpeg -q

In [ ]:
from google.colab import drive
import roboflow
import cv2
import os
import time
from pathlib import Path
from tqdm import tqdm
import imageio

drive.mount('/content/drive')

In [ ]:
def get_extracted_filenames(output_dir: str) -> set:
    if not os.path.exists(output_dir):
        return set()

    existing = {
        f for f in os.listdir(output_dir)
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))
    }
    print(f"  → {len(existing)} file sudah ada di output_dir")
    return existing

In [ ]:
def get_frames_to_extract(total_frames: int, every_n_frames: int, fmt: str) -> dict:
    frames = {}
    extracted_count = 0
    for frame_count in range(total_frames):
        if frame_count % every_n_frames == 0:
            fname = f"frame_{extracted_count:06d}.{fmt}"
            frames[fname] = frame_count
            extracted_count += 1
    return frames

In [ ]:
def get_frames_to_skip(all_frames: dict, existing_files: set) -> dict:
    to_extract = {
        fname: idx
        for fname, idx in all_frames.items()
        if fname not in existing_files
    }
    already_exists = len(all_frames) - len(to_extract)

    print(f"\n  Ringkasan:")
    print(f"  Sudah diekstrak : {already_exists} frame")
    print(f"  Belum diekstrak : {len(to_extract)} frame")
    return to_extract

In [ ]:
def extract_frames_opencv(video_path: str, output_dir: str, to_extract: dict,
                          fmt: str = 'jpg', quality: int = 95) -> bool:
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Error: Tidak bisa membuka video {video_path}")
        return False

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS)
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"\n📹 Informasi Video:")
    print(f"   Total Frames : {total_frames}")
    print(f"   FPS          : {fps}")
    print(f"   Resolusi     : {width}x{height}")
    print(f"   Durasi       : {total_frames/fps:.2f} detik")

    target_indices = {idx: fname for fname, idx in to_extract.items()}

    extracted, skipped = 0, 0
    frame_count = 0

    with tqdm(total=total_frames, desc="🎬 Extracting") as pbar:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_count in target_indices:
                fname    = target_indices[frame_count]
                filepath = os.path.join(output_dir, fname)

                if fmt.lower() == 'png':
                    cv2.imwrite(filepath, frame, [cv2.IMWRITE_PNG_COMPRESSION, 9])
                elif fmt.lower() == 'jpg':
                    cv2.imwrite(filepath, frame, [cv2.IMWRITE_JPEG_QUALITY, quality])

                extracted += 1
            else:
                skipped += 1

            frame_count += 1
            pbar.update(1)

    cap.release()

    print(f"\n✅ Selesai!")
    print(f"   Diekstrak : {extracted} frame")
    print(f"   Dilewati  : {skipped} frame (sudah ada / tidak perlu)")
    print(f"   Output    : {output_dir}")
    return True

In [ ]:
def main_extract_frame(video_path: str, output_dir: str, every_n_frames: int,
         fmt: str, quality: int) -> None:

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Error: Tidak bisa membuka video {video_path}")
        return
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    print("📂 Mengecek file yang sudah ada di output_dir...")
    existing_files = get_extracted_filenames(output_dir)

    print("\n🔢 Menghitung frame yang akan diekstrak...")
    all_frames = get_frames_to_extract(total_frames, every_n_frames, fmt)

    to_extract = get_frames_to_skip(all_frames, existing_files)

    if not to_extract:
        print("\n✅ Semua frame sudah diekstrak, tidak ada yang perlu diproses.")
        return

    extract_frames_opencv(video_path, output_dir, to_extract, fmt, quality)

In [ ]:
def _search_batch_page(project, batch_id: str, offset: int, per_page: int,
                        fields: list, max_retries: int = 5, base_delay: float = 2.0):
    """
    Panggil project.search() untuk satu halaman dengan retry + exponential backoff.
    Ini PENTING karena jika satu request gagal (timeout/rate-limit) di tengah
    pagination ribuan foto, kita tidak boleh langsung berhenti (break) karena
    itu akan membuat daftar nama file yang ter-fetch menjadi TIDAK LENGKAP,
    yang berakibat file yang sebenarnya sudah ada di Roboflow dianggap
    'belum diupload' lalu diupload ulang (duplikat).

    Return:
        list hasil, atau None jika semua percobaan gagal (pemanggil harus
        memutuskan apakah mau stop atau lanjut skip halaman ini).
    """
    for attempt in range(1, max_retries + 1):
        try:
            results = project.search(
                batch=True,
                batch_id=batch_id,
                offset=offset,
                limit=per_page,
                fields=fields,
            )
            if isinstance(results, dict):
                results = results.get("results", [])
            return results
        except Exception as e:
            if attempt == max_retries:
                print(f"  [ERROR] Gagal di offset {offset} setelah {max_retries}x percobaan: {e}")
                return None
            delay = base_delay * (2 ** (attempt - 1))
            print(f"  [WARN] Gagal di offset {offset} (percobaan {attempt}/{max_retries}): {e}. "
                  f"Retry dalam {delay:.1f}s...")
            time.sleep(delay)
    return None

In [ ]:
def fetch_all_batch_items(project, batch_id: str, total_images: int, fields: list,
                           per_page: int = 100) -> tuple[list, bool]:
    """
    Ambil SEMUA item dalam sebuah batch dengan pagination offset/limit.

    ⚠️ PENTING soal 'limit' yang diminta ke API:
    project.search() TIDAK menjamin akan mengembalikan hasil sebanyak `limit`
    yang diminta — API bisa saja mem-batasi (cap) jumlah hasil per-request ke
    angka lebih kecil dari yang diminta (mis. diminta limit=1000 tapi hanya
    mengembalikan 100). Karena itu, KITA TIDAK BOLEH memakai
    'len(results) < per_page' sebagai syarat 'ini halaman terakhir', karena
    itu bisa membuat pagination berhenti prematur setelah halaman PERTAMA
    (bug yang sempat terjadi: data terfetch jadi jauh lebih sedikit dari
    total_images yang sebenarnya).

    Aturan berhenti yang BENAR dipakai di sini:
      - offset selalu bertambah sebesar JUMLAH HASIL YANG BENAR-BENAR
        DITERIMA (len(results)), bukan sebesar per_page yang diminta.
      - loop berhenti hanya jika: hasil kosong (results == []), ATAU total
        yang sudah ter-fetch >= total_images (safety net agar tidak infinite
        loop bila offset tidak lagi bertambah karena results kosong terus).

    ⚠️ CATATAN soal keandalan deteksi 'duplikat nama file':
    Selain masalah limit di atas, urutan hasil antar-request pagination juga
    tidak dijamin stabil (deterministic), terutama ketika banyak item punya
    timestamp 'created' yang sama/berdekatan (kasus umum saat upload massal
    lewat script). Ini bisa membuat algoritma deteksi duplikat berbasis 'nama
    muncul >1 kali di hasil fetch' memberi angka yang TIDAK MEREPRESENTASIKAN
    duplikat asli di Roboflow (false positive) — sudah diverifikasi manual
    oleh pengguna bahwa batch tidak memiliki duplikat nama sungguhan. Karena
    itu, hasil 'duplikat' dari fungsi-fungsi terkait TIDAK BOLEH dipakai
    sebagai dasar untuk MENGHAPUS data tanpa verifikasi manual.

    Return:
        (items, complete) — items: list semua hasil yang berhasil ter-fetch.
        complete: True jika seluruh pagination selesai tanpa error fatal.
    """
    items = []
    offset = 0

    while True:
        results = _search_batch_page(project, batch_id, offset, per_page, fields=fields)

        if results is None:
            print(f"  ⚠️  PERINGATAN: Gagal mengambil data di offset {offset} setelah retry. "
                  f"Data KEMUNGKINAN TIDAK LENGKAP ({len(items)}/{total_images} terfetch).")
            return items, False

        if not results:
            break

        items.extend(results)
        print(f"  [Offset {offset:>5}] → terfetch: {len(items)}/{total_images}")

        # Majukan offset sebesar jumlah hasil yang BENAR-BENAR diterima,
        # bukan sebesar per_page yang diminta (API bisa cap limit lebih kecil).
        offset += len(results)

        # Safety net: kalau total ter-fetch sudah >= total_images (jika info
        # ini tersedia dan valid), tidak perlu lanjut request lagi.
        if total_images and len(items) >= total_images:
            break

    return items, True

In [ ]:
def get_roboflow_filenames(project, batch_name: str) -> tuple[set, int]:
    """
    Return:
        existing_filenames : set nama unik di Roboflow
        total_roboflow     : total foto di Roboflow (termasuk kemungkinan duplikat)

    Catatan: angka 'duplikat' yang dicetak di sini bersifat INDIKATIF saja
    (lihat penjelasan di fetch_all_batch_items) — karena itu fungsi ini HANYA
    dipakai untuk membangun existence-check (mana file yang perlu diupload),
    BUKAN untuk memutuskan penghapusan data.
    """
    existing_filenames = set()
    batch_id = None
    total_images = 0

    # ── 1. Ambil batch_id ──────────────────────────────────────────────────
    batches = project.get_batches().get("batches", [])
    for batch in batches:
        if batch.get("name") == batch_name:
            batch_id = batch.get("id")
            total_images = batch.get("images", 0)
            break

    if not batch_id:
        print(f"  [ERROR] Batch '{batch_name}' tidak ditemukan.")
        return existing_filenames, 0

    print(f"  → Batch ditemukan: id={batch_id}, total={total_images} foto")

    # ── 2. Ambil semua nama file dengan pagination (retry, offset dinamis) ─
    items, complete = fetch_all_batch_items(project, batch_id, total_images, fields=["name"])

    all_names = []
    for item in items:
        name = item.get("name") or item.get("filename") or ""
        name = os.path.basename(name)
        all_names.append(name)
        existing_filenames.add(name)

    if not complete:
        print("  ⚠️  Data existing_filenames TIDAK LENGKAP karena pagination gagal di tengah jalan. "
              "Sebaiknya jalankan ulang fungsi ini sebelum melanjutkan ke upload.")

    # ── 3. Laporan (indikatif) ──────────────────────────────────────────────
    duplicate_count = len(all_names) - len(existing_filenames)
    print(f"\n  📊 Laporan Roboflow:")
    print(f"     Terfetch total       : {len(all_names)}")
    print(f"     Nama unik            : {len(existing_filenames)}")
    print(f"     Duplikat (indikatif) : {duplicate_count}  "
          f"(⚠️ angka ini bisa TIDAK akurat karena urutan API tidak stabil; "
          f"jangan dipakai untuk memutuskan penghapusan)")

    return existing_filenames, total_images

In [ ]:
def get_local_filenames(output_dir: str) -> dict:
    local_files = {
        filename: os.path.join(output_dir, filename)
        for filename in os.listdir(output_dir)
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))
    }
    print(f"  → {len(local_files)} file ditemukan di lokal")
    return local_files

In [ ]:
def get_files_to_upload(local_files: dict, existing_filenames: set, total_roboflow: int) -> dict:
    """Kembalikan file lokal yang belum ada di Roboflow."""
    to_upload = {
        filename: path
        for filename, path in local_files.items()
        if filename not in existing_filenames
    }
    already_uploaded = local_files.keys() & existing_filenames

    print(f"\n  📊 Ringkasan:")
    print(f"     File lokal              : {len(local_files)}")
    print(f"     Sudah di Roboflow       : {total_roboflow} (termasuk kemungkinan duplikat)")
    print(f"     Nama unik di Roboflow   : {len(existing_filenames)}")
    print(f"     Match lokal ↔ Roboflow  : {len(already_uploaded)}")
    print(f"     Belum diupload          : {len(to_upload)}")
    return to_upload

In [ ]:
def upload_files_to_roboflow(project, to_upload: dict, batch_name: str) -> None:
    uploaded, failed = 0, 0

    for filename, image_path in to_upload.items():
        print(f"  [UPLOAD] {filename}")
        try:
            project.upload(
                image_path=image_path,
                batch_name=batch_name,
                num_retry_uploads=3
            )
            uploaded += 1
        except Exception as e:
            print(f"  [ERROR]  {filename} → {e}")
            failed += 1

    print(f"\nSelesai → Berhasil: {uploaded} | Gagal: {failed} | Dilewati: {len(to_upload) - uploaded}")

In [ ]:
def delete_roboflow_duplicates(project, batch_name: str, i_understand_the_risk: bool = False) -> None:
    """
    ⚠️⚠️⚠️ DINONAKTIFKAN SECARA DEFAULT — BACA SEBELUM MENGGUNAKAN ⚠️⚠️⚠️

    Fungsi ini awalnya dimaksudkan untuk menghapus foto duplikat di sebuah
    batch Roboflow, dengan cara: fetch semua item lewat pagination
    offset/limit, lalu anggap nama yang muncul >1 kali sebagai duplikat.

    MASALAH YANG DITEMUKAN: project.search() TIDAK menjamin urutan hasil
    stabil (deterministic) antar-request pagination, terutama ketika banyak
    item memiliki timestamp 'created' yang sama/berdekatan (kasus umum saat
    upload massal lewat script seperti notebook ini). Akibatnya:
      - Nama file yang SAMA bisa muncul di 2 halaman berbeda karena urutan
        bergeser di sekitar boundary offset → dianggap 'duplikat' padahal
        di Roboflow cuma ada 1 foto dengan nama itu (FALSE POSITIVE).
      - Ini sudah diverifikasi manual oleh pengguna: batch TIDAK memiliki
        duplikat nama sungguhan, namun laporan duplikat dari fungsi ini
        berubah-ubah setiap dijalankan, yang membuktikan angka tersebut
        TIDAK RELIABLE.
      - Karena 'duplicate_ids' yang dihasilkan tidak reliable, ada risiko
        NYATA menghapus foto yang BUKAN duplikat.

    Sampai ada mekanisme pagination yang benar-benar stabil (mis. sort key
    eksplisit yang dijamin API), fungsi ini TIDAK BOLEH dipakai untuk
    menghapus data secara otomatis.

    Jika Anda tetap ingin menjalankannya untuk INSPEKSI (tanpa menghapus),
    set i_understand_the_risk=True. Fungsi tetap TIDAK akan menghapus
    apa pun secara otomatis — laporan yang dihasilkan hanya untuk referensi
    manual, dan setiap penghapusan HARUS diverifikasi satu per satu di
    Roboflow UI sebelum dieksekusi.
    """
    if not i_understand_the_risk:
        print("  ⛔ Fungsi ini dinonaktifkan karena deteksi duplikat berbasis pagination "
              "offset/limit TERBUKTI TIDAK RELIABLE (angka duplikat berubah-ubah antar-run "
              "padahal tidak ada duplikat nama sungguhan di batch).\n"
              "  Tidak ada penghapusan yang akan dilakukan.\n"
              "  Jika Anda memahami risikonya dan hanya ingin melihat laporan (tanpa hapus), "
              "panggil ulang dengan i_understand_the_risk=True.")
        return

    batch_id = None
    total_images = 0

    batches = project.get_batches().get("batches", [])
    for batch in batches:
        if batch.get("name") == batch_name:
            batch_id = batch.get("id")
            total_images = batch.get("images", 0)
            break

    if not batch_id:
        print(f"  [ERROR] Batch '{batch_name}' tidak ditemukan.")
        return

    print(f"  → Batch ditemukan: id={batch_id}, total={total_images} foto")

    items, complete = fetch_all_batch_items(project, batch_id, total_images, fields=["id", "name"])

    if not complete:
        print("  ⛔ Pagination gagal di tengah jalan. Data tidak lengkap — "
              "proses dihentikan demi keamanan, TIDAK ADA yang dihapus.")
        return

    seen_names = {}
    duplicate_candidates = []  # (name, image_id) — hanya KANDIDAT, belum tentu duplikat asli

    for item in items:
        name = os.path.basename(item.get("name") or item.get("filename") or "")
        image_id = item.get("id")

        if not image_id:
            print(f"  [WARN] Item '{name}' tidak punya id, dilewati.")
            continue

        if name in seen_names:
            duplicate_candidates.append((name, image_id))
        else:
            seen_names[name] = image_id

    print(f"\n  📊 Laporan Kandidat Duplikat (BELUM TENTU AKURAT):")
    print(f"     Total terfetch      : {len(items)}")
    print(f"     Nama unik           : {len(seen_names)}")
    print(f"     Kandidat duplikat   : {len(duplicate_candidates)}")

    if not duplicate_candidates:
        print("\n  ✅ Tidak ada kandidat duplikat.")
        return

    print("\n  ⚠️  Fungsi ini TIDAK akan menghapus apa pun secara otomatis.")
    print("  Daftar kandidat (name, image_id) untuk diverifikasi MANUAL di Roboflow UI:")
    for name, image_id in duplicate_candidates:
        print(f"    - {name} → {image_id}")

    print("\n  ℹ️  Jika setelah verifikasi manual di UI Roboflow Anda yakin item-item di atas "
          "benar-benar duplikat, hapus secara manual satu per satu, atau gunakan "
          "project.delete_images([...]) dengan daftar id yang SUDAH DIVERIFIKASI, "
          "bukan hasil mentah dari fungsi ini.")

In [ ]:
def main_upload_to_roboflow(project, output_dir: str, batch_name: str) -> None:
    existing_filenames, total_roboflow = get_roboflow_filenames(project, batch_name)
    local_files = get_local_filenames(output_dir)

    to_upload = get_files_to_upload(local_files, existing_filenames, total_roboflow)

    if not to_upload:
        print("\n✅ Semua file sudah ada di Roboflow, tidak ada yang perlu diupload.")
        return

    print("\nMemulai upload...")
    upload_files_to_roboflow(project, to_upload, batch_name)

In [ ]:
rf = roboflow.Roboflow(api_key="hRSsLIOXzzSZPkM0Wd54")
project = rf.workspace().project("plastic-trash-detection-p9gqv")
filename = "0412"
MAIN_PATH = "/content/drive/MyDrive/"
# VIDEO_PATH = f"{MAIN_PATH}/DJI_{filename}.MOV" # bisa .MOV atau .MP4
OUTPUT_DIR = f"{MAIN_PATH}/DJISampah/{filename}"
EXTRACT_EVERY_N_FRAMES = 1  # 1 = semua frame, 2 = setiap frame ke-2, dst
OUTPUT_FORMAT = 'jpg'        # 'png' (lossless) atau 'jpg' (lossy)
QUALITY = 98                 # Untuk JPG (1-100)

In [ ]:
# main_extract_frame(VIDEO_PATH, OUTPUT_DIR, EXTRACT_EVERY_N_FRAMES, OUTPUT_FORMAT, QUALITY)

In [ ]:
# Fungsi ini sekarang HANYA menampilkan kandidat duplikat untuk diverifikasi manual,
# tidak lagi otomatis menghapus apa pun (lihat docstring delete_roboflow_duplicates).
# delete_roboflow_duplicates(project, batch_name=filename, i_understand_the_risk=True)

In [ ]:
main_upload_to_roboflow(project, OUTPUT_DIR, filename)